# Creacion de Ventanas Deslizantes con metodo por Transecto y metodo general

In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
Código 4 (adaptado a transectos) - VERSIÓN MEJORADA
Creación de ventanas deslizantes de 72h con verificación de dimensionalidad.
"""

import os
import numpy as np
import pandas as pd
from pathlib import Path

# ============================================================================
# CONFIGURACIÓN
# ============================================================================

BASE_DIR = os.path.expanduser("/usu/snsaetor/Documents/GitHub/TFGFinal/Datos_TFG_outliers/")
INPUT_ML_TRANSECT = os.path.join(BASE_DIR, "encoded", "ml", "by_transect")
INPUT_DL_TRANSECT = os.path.join(BASE_DIR, "encoded", "dl", "by_transect")
INPUT_ML_GLOBAL = os.path.join(BASE_DIR, "encoded", "ml", "global")
INPUT_DL_GLOBAL = os.path.join(BASE_DIR, "encoded", "dl", "global")

OUTPUT_WINDOWS = os.path.join(BASE_DIR, "windows")
OUTPUT_ML_TRANSECT = os.path.join(OUTPUT_WINDOWS, "by_transect", "ml")
OUTPUT_DL_TRANSECT = os.path.join(OUTPUT_WINDOWS, "by_transect", "dl")
OUTPUT_ML_GLOBAL = os.path.join(OUTPUT_WINDOWS, "global", "ml")
OUTPUT_DL_GLOBAL = os.path.join(OUTPUT_WINDOWS, "global", "dl")

for path in [OUTPUT_ML_TRANSECT, OUTPUT_DL_TRANSECT, OUTPUT_ML_GLOBAL, OUTPUT_DL_GLOBAL]:
    os.makedirs(path, exist_ok=True)

WINDOW_IN = 72
WINDOW_OUT = 72
TARGET_COL = 'O3'

# ============================================================================
# FUNCIONES AUXILIARES
# ============================================================================

def create_windows_with_timestamps(df, target_col=TARGET_COL, window_in=WINDOW_IN, window_out=WINDOW_OUT):
    if not isinstance(df.index, pd.DatetimeIndex):
        raise ValueError("El DataFrame debe tener índice DatetimeIndex")
    df_sorted = df.sort_index()
    data = df_sorted.values
    n_features = data.shape[1]
    feature_names = df_sorted.columns.tolist()

    n = len(df_sorted)
    X_ml_list = []
    X_dl_list = []
    y_list = []
    timestamps = []

    for i in range(window_in, n - window_out):
        in_data = data[i - window_in:i, :]
        out_data = df_sorted[target_col].iloc[i:i + window_out].values
        X_dl_list.append(in_data)
        X_ml_list.append(in_data.flatten())
        y_list.append(out_data)
        timestamps.append(df_sorted.index[i])

    if len(X_ml_list) == 0:
        return None, None, None, None, feature_names

    X_ml = np.array(X_ml_list, dtype=np.float32)
    X_dl = np.array(X_dl_list, dtype=np.float32)
    y = np.array(y_list, dtype=np.float32)
    timestamps = np.array(timestamps, dtype='datetime64[h]')

    return X_ml, X_dl, y, timestamps, feature_names

def process_file_ml_dl(ml_path, dl_path, output_ml_dir, output_dl_dir, name):
    print(f"  Procesando {name}...")

    # Cargar y filtrar numéricas
    df_ml = pd.read_csv(ml_path, index_col=0, parse_dates=True)
    df_dl = pd.read_csv(dl_path, index_col=0, parse_dates=True)

    df_ml = df_ml.select_dtypes(include=[np.number])
    df_dl = df_dl.select_dtypes(include=[np.number])

    # Verificar que no estén vacíos
    if df_ml.shape[1] == 0:
        print(f"    Error: {ml_path} no tiene columnas numéricas. Se omite.")
        return
    if df_dl.shape[1] == 0:
        print(f"    Error: {dl_path} no tiene columnas numéricas. Se omite.")
        return

    # Alinear índices
    if not df_ml.index.equals(df_dl.index):
        print(f"    Advertencia: índices no coinciden. Alineando con df_ml.")
        df_dl = df_dl.reindex(df_ml.index)

    # Generar ventanas con ML (para y y timestamps)
    result = create_windows_with_timestamps(df_ml)
    if result[0] is None:
        print(f"    No se generaron ventanas para {name}")
        return
    X_ml, _, y, timestamps, feat_names_ml = result

    # Generar ventanas con DL (para X_dl)
    res_dl = create_windows_with_timestamps(df_dl)
    if res_dl[0] is None:
        print(f"    No se generaron ventanas DL para {name}")
        return
    X_dl, _, _, _, _ = res_dl

    # Verificar número de ventanas
    if len(X_ml) != len(X_dl):
        print(f"    Error: número de ventanas ML ({len(X_ml)}) y DL ({len(X_dl)}) difiere. Se omite.")
        return

    # --- VERIFICACIÓN DE DIMENSIONALIDAD ---
    if X_dl.ndim != 3:
        print(f"    ERROR: X_dl tiene {X_dl.ndim} dimensiones. Se esperaban 3. Shape: {X_dl.shape}")
        # Opcional: intentar reparar si es 2D (aplanado) asumiendo que window_in=72
        if X_dl.ndim == 2 and X_dl.shape[1] % WINDOW_IN == 0:
            n_samples = X_dl.shape[0]
            n_features = X_dl.shape[1] // WINDOW_IN
            X_dl = X_dl.reshape(n_samples, WINDOW_IN, n_features)
            print(f"      Reparado: ahora es {X_dl.shape}")
        else:
            return
    # ---------------------------------------

    # Guardar archivos ML
    np.save(os.path.join(output_ml_dir, f"{name}_X.npy"), X_ml)
    np.save(os.path.join(output_ml_dir, f"{name}_y.npy"), y)
    np.save(os.path.join(output_ml_dir, f"{name}_timestamps.npy"), timestamps)

    # Guardar archivos DL
    np.save(os.path.join(output_dl_dir, f"{name}_X.npy"), X_dl)
    np.save(os.path.join(output_dl_dir, f"{name}_y.npy"), y)
    np.save(os.path.join(output_dl_dir, f"{name}_timestamps.npy"), timestamps)

    print(f"    Ventanas guardadas: {len(X_ml)} muestras. Shapes: X_ml {X_ml.shape}, X_dl {X_dl.shape}")

# ============================================================================
# PROCESAMIENTO POR TRANSECTO Y GLOBAL (igual que antes)
# ============================================================================

def process_by_transect():
    print("\n--- Procesando datos por transecto ---")
    if not os.path.exists(INPUT_ML_TRANSECT) or not os.path.exists(INPUT_DL_TRANSECT):
        print("  Las carpetas de entrada por transecto no existen.")
        return
    ml_files = {f.stem: f for f in Path(INPUT_ML_TRANSECT).glob("*.csv")}
    dl_files = {f.stem: f for f in Path(INPUT_DL_TRANSECT).glob("*.csv")}
    common = set(ml_files.keys()) & set(dl_files.keys())
    if not common:
        print("  No hay archivos coincidentes entre ML y DL.")
        return
    for name in sorted(common):
        process_file_ml_dl(ml_files[name], dl_files[name],
                           OUTPUT_ML_TRANSECT, OUTPUT_DL_TRANSECT, name)

def process_global():
    print("\n--- Procesando datos globales (por estación) ---")
    if not os.path.exists(INPUT_ML_GLOBAL) or not os.path.exists(INPUT_DL_GLOBAL):
        print("  Las carpetas de entrada global no existen.")
        return
    ml_files = {f.stem: f for f in Path(INPUT_ML_GLOBAL).glob("*.csv")}
    dl_files = {f.stem: f for f in Path(INPUT_DL_GLOBAL).glob("*.csv")}
    common = set(ml_files.keys()) & set(dl_files.keys())
    if not common:
        print("  No hay archivos coincidentes entre ML y DL global.")
        return
    for name in sorted(common):
        process_file_ml_dl(ml_files[name], dl_files[name],
                           OUTPUT_ML_GLOBAL, OUTPUT_DL_GLOBAL, name)

if __name__ == "__main__":
    print("=" * 60)
    print("Creación de ventanas deslizantes (72h in / 72h out) - Versión mejorada")
    print("=" * 60)
    process_by_transect()
    process_global()
    print("\nProceso completado. Revise la carpeta:")
    print(f"  {OUTPUT_WINDOWS}")

Creación de ventanas deslizantes (72h in / 72h out) - Versión mejorada

--- Procesando datos por transecto ---
  Procesando Transecto_1...
    ERROR: X_dl tiene 2 dimensiones. Se esperaban 3. Shape: (34898, 1656)
      Reparado: ahora es (34898, 72, 23)
    Ventanas guardadas: 34898 muestras. Shapes: X_ml (34898, 1512), X_dl (34898, 72, 23)
  Procesando Transecto_2...
    ERROR: X_dl tiene 2 dimensiones. Se esperaban 3. Shape: (34898, 1656)
      Reparado: ahora es (34898, 72, 23)
    Ventanas guardadas: 34898 muestras. Shapes: X_ml (34898, 1512), X_dl (34898, 72, 23)

--- Procesando datos globales (por estación) ---
  Procesando T1_E1_Alicante...
    ERROR: X_dl tiene 2 dimensiones. Se esperaban 3. Shape: (17377, 1656)
      Reparado: ahora es (17377, 72, 23)
    Ventanas guardadas: 17377 muestras. Shapes: X_ml (17377, 1512), X_dl (17377, 72, 23)
  Procesando T1_E2_Elda...
    ERROR: X_dl tiene 2 dimensiones. Se esperaban 3. Shape: (17377, 1656)
      Reparado: ahora es (17377, 72, 23